# bfs-gpu on Google Colab (Group 17)

Builds and checks the project on a Colab GPU. **Before running:** *Runtime -> Change runtime type -> T4 GPU*,
then *Runtime -> Run all*. Each step says what output to expect.

Getting the code in (step 2): either upload `bfs-gpu.zip` (made with `python scripts/make_zip.py` in the repo),
or, once the team repository exists on GitHub, clone it.

**Warning (playbook 4.3):** Colab GPUs differ between sessions and the disk is wiped when the session ends.
Use any Colab session for development. If Colab is the reference machine, run the *final* experiments in
**one single session**, and save `results/` (including `env.txt` and `gpu_info.txt`) to Google Drive or
download it before closing.

## 1. GPU and toolchain

In [ ]:
!nvidia-smi
!nvcc --version | tail -2
!g++ --version | head -1

## 2. Get the code
Run **one** of the two cells. Option A asks you to pick `bfs-gpu.zip` from your computer.

In [ ]:
# Option A: upload bfs-gpu.zip
import os, shutil, zipfile
from google.colab import files
os.chdir('/content')
up = files.upload()                      # choose bfs-gpu.zip
name = next(iter(up))
shutil.rmtree('/content/bfs-gpu', ignore_errors=True)
zipfile.ZipFile(name).extractall('/content/bfs-gpu')
print('extracted', name, '->', '/content/bfs-gpu')

In [ ]:
# Option B: clone the team repository (fill in the URL; private repos need a token, never commit it)
# !rm -rf /content/bfs-gpu && git clone https://<token>@github.com/<member1-user>/bfs-gpu.git /content/bfs-gpu

## 3. Pick the CUDA architecture of this GPU
A T4 is compute capability 7.5, so `ARCH=sm_75`. `make` reads `ARCH` from the environment, so every later
step (including `scripts/check_all.sh`) builds for this GPU.

In [ ]:
import os, subprocess
cap = subprocess.check_output(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader']).decode().split()[0]
os.environ['ARCH'] = 'sm_' + cap.replace('.', '')
os.chdir('/content/bfs-gpu')
print('GPU compute capability', cap, '-> ARCH =', os.environ['ARCH'])

## 4. Host build and unit tests (Member 1)
Expected last line: `39 checks passed, 0 failed`.

In [ ]:
%%bash
cd /content/bfs-gpu
make clean > /dev/null
make host 2>&1 | grep -iE "warning|error" || echo "make host: no warnings"
./bin/test_host

## 5. Toolchain check: `hello_cuda`
Expected: a `GPU: ...` line, then `hello_cuda: PASS`.

In [ ]:
%%bash
cd /content/bfs-gpu
nvcc -O2 -std=c++17 -arch=$ARCH -Iinclude tests/hello_cuda.cu -o /tmp/hello && /tmp/hello

## 6. Golden values (playbook 6.7)
Expected: `n=262144 m=7612566`; stats show 88105 isolated vertices, max degree 25096, CV 6.473,
est. diameter 7; serial first line `src=149623 ... levels=6 edges=7612472 ... structural=OK`.

In [ ]:
%%bash
cd /content/bfs-gpu
./bin/graph_tool rmat 18 16 1 /tmp/r18.csr
./bin/graph_tool stats /tmp/r18.csr | sed -n '1,11p'
./bin/graph_tool serial /tmp/r18.csr 2

## 7. Full check: `scripts/check_all.sh`
While teammates' files are still placeholders, this stops at step 4 (`TEST_NAIVE: FAIL (placeholder)`):
that is expected, and step 0 lists the placeholder files. Once Members 2 and 3 have delivered, the expected
last line is `ALL CHECKS PASSED`.

In [ ]:
%%bash
cd /content/bfs-gpu
bash scripts/check_all.sh 2>&1 | grep -vE "^(g\+\+|nvcc) "

## 8. Smoke test of the benchmark (needs Member 2's `bench.cu` and both GPU programs)
Expected last line: `all runs verified OK -> results/smoke.csv`.

In [ ]:
%%bash
cd /content/bfs-gpu
mkdir -p data/csr results
[ -f data/csr/rmat16.csr ] || ./bin/graph_tool rmat 16 16 1 data/csr/rmat16.csr
./bin/bench --graph data/csr/rmat16.csr --category rmat --sources 2 --trials 2 \
            --out results/smoke.csv --structural

## 9. (Optional) Datasets
Downloads about 2 GB from SNAP and converts everything (roughly 10-15 minutes). Colab's disk is wiped at
the end of the session; to keep the `.csr` files, mount Drive in the next cell first. Converting com-orkut
needs about 4.5 GB of RAM.

In [ ]:
# Optional: keep data/ on Google Drive across sessions (asks for permission to access your Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/bfs-gpu-data && rm -rf /content/bfs-gpu/data \
#   && ln -s /content/drive/MyDrive/bfs-gpu-data /content/bfs-gpu/data

In [ ]:
%%bash
cd /content/bfs-gpu
bash scripts/download_datasets.sh 2>&1 | grep -E "^\[|done"
bash scripts/convert_all.sh 2>&1 | grep -E "^\[|parsed|generated|missing"

## 10. Record the machine (do this in the session that produces final numbers)
Writes `results/env.txt` and `results/gpu_info.txt`, which must sit next to the final CSVs.

In [ ]:
%%bash
cd /content/bfs-gpu
mkdir -p results
bash scripts/env_report.sh > results/env.txt
nvidia-smi > results/gpu_info.txt
[ -x bin/bench ] && ./bin/bench --gpu-info >> results/gpu_info.txt || true
cat results/env.txt

## 11. Save results
Downloads `results/` as a zip (CSV files, figures, env.txt, gpu_info.txt).

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/results', 'zip', '/content/bfs-gpu/results')
files.download('/content/results.zip')